In [87]:
"""
ai_picks_pipeline.py

Ensemble AI horse racing picks (Claude / Gemini) with morning line odds,
convert to win probabilities, and prepare for backtesting.

Data structure:
    data/ai_picks/{date}/claude.csv     - AI estimated Beyers
    data/ai_picks/{date}/gemini.csv     - AI estimated Beyers
    data/upc_entries/{track}.csv        - Entries with ML odds
    data/entries/*.csv                  - Historical results

AI pick CSV format:
    race, horse_number, expected_beyer, track

Results CSV format (data/entries/*.csv):
    running, horse, horse_name, sire, trainer, jockey, ml_odds,
    finish, win_paid, place_paid, show_paid, track, date, race
"""

import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import log_loss

In [88]:
def load_ai_picks(date_str, base_dir='data/ai_picks'):
    """
    Load Claude and Gemini pick CSVs for a given date.
    Each CSV has: race, horse_number, expected_beyer, track
    
    Returns merged DataFrame with claude_beyer, gemini_beyer columns.
    """
    date_dir = os.path.join(base_dir, date_str)
    
    picks = {}
    for source in ['claude', 'gemini']:
        path = os.path.join(date_dir, f'{source}.csv')
        if os.path.exists(path):
            df = pd.read_csv(path)
            
            # Normalize column names - find the beyer column
            cols = df.columns.tolist()
            beyer_col = [c for c in cols if 'beyer' in c.lower() or 'expected' in c.lower()]
            if beyer_col:
                df = df.rename(columns={beyer_col[0]: f'{source}_beyer'})
            else:
                # If no track column, assume 3 cols: race, horse_number, beyer
                if len(cols) == 3:
                    df.columns = ['race', 'horse_number', f'{source}_beyer']
                elif len(cols) == 4:
                    df.columns = ['race', 'horse_number', f'{source}_beyer', 'track']
            
            df['race'] = df['race'].astype(int)
            df['horse_number'] = df['horse_number'].astype(str).str.strip()
            
            keep_cols = ['race', 'horse_number', f'{source}_beyer']
            if 'track' in df.columns:
                keep_cols.append('track')
            
            picks[source] = df[keep_cols]
    
    if not picks:
        raise FileNotFoundError(f"No AI pick files found in {date_dir}")
    
    # Merge on race + horse_number (and track if present)
    sources = list(picks.keys())
    merged = picks[sources[0]]
    for source in sources[1:]:
        merge_keys = ['race', 'horse_number']
        if 'track' in merged.columns and 'track' in picks[source].columns:
            merge_keys.append('track')
        merged = merged.merge(picks[source], on=merge_keys, how='outer')
    
    merged['date'] = date_str
    
    # Default track if not present
    if 'track' not in merged.columns:
        merged['track'] = 'keeneland'  # sensible default for Kee card
    
    return merged

In [89]:
def load_entries(base_dir='data/entries', track=None, date=None):
    """
    Load historical results from data/entries/*.csv
    
    Format: running, horse, horse_name, sire, trainer, jockey, ml_odds,
            finish, win_paid, place_paid, show_paid, track, date, race
    """
    if not os.path.exists(base_dir):
        return pd.DataFrame()
    
    results_files = [f for f in os.listdir(base_dir) if f.endswith('.csv')]
    
    if not results_files:
        return pd.DataFrame()
    
    dfs = []
    for f in results_files:
        try:
            df = pd.read_csv(os.path.join(base_dir, f))
            dfs.append(df)
        except Exception as e:
            print(f"Warning: Could not load {f}: {e}")
    
    if not dfs:
        return pd.DataFrame()
    
    results = pd.concat(dfs, ignore_index=True)
    
    # Only running horses
    if 'running' in results.columns:
        results = results[results['running'] == 1].copy()
    
    # Parse date
    if 'date' in results.columns:
        results['date'] = pd.to_datetime(results['date']).dt.strftime('%Y-%m-%d')
    
    # Filter
    if track is not None:
        results = results[results['track'] == track]
    if date is not None:
        results = results[results['date'] == date]
    
    # Derived columns
    if 'finish' in results.columns:
        results['won'] = (results['finish'] == 1).astype(int)
        results['place'] = (results['finish'] <= 2).astype(int)
        results['show'] = (results['finish'] <= 3).astype(int)
    
    # Payout per dollar
    if 'win_paid' in results.columns:
        results['payout_per_dollar'] = pd.to_numeric(
            results['win_paid'].astype(str).str.replace('$', '').str.replace(',', ''),
            errors='coerce'
        ).fillna(0) / 2  # $2 base
    
    # Create horse_number from 'horse' column if needed
    # Keep as string to handle coupled entries like '1A', '1B'
    if 'horse' in results.columns and 'horse_number' not in results.columns:
        results['horse_number'] = results['horse'].astype(str).str.strip()
    
    return results.reset_index(drop=True)
 
 
def load_upc_entries(track, base_dir='data/upc_entries'):
    """
    Load upcoming/current entries with ML odds from data/upc_entries/{track}.csv
    """
    path = os.path.join(base_dir, f'{track}.csv')
    if not os.path.exists(path):
        return None
    
    df = pd.read_csv(path)
    return df
 

def load_all_ai_dates(base_dir='../data/ai_picks'):
    """Load and concatenate all available AI pick dates."""
    if not os.path.exists(base_dir):
        return pd.DataFrame()
    
    dates = sorted([d for d in os.listdir(base_dir) 
                    if os.path.isdir(os.path.join(base_dir, d))])
    
    all_picks = []
    for date_str in dates:
        try:
            picks = load_ai_picks(date_str, base_dir)
            all_picks.append(picks)
        except FileNotFoundError:
            continue
    
    return pd.concat(all_picks, ignore_index=True) if all_picks else pd.DataFrame()



In [90]:
# =============================================================================
# 2. BEYER COMPOSITE & BLENDING
# =============================================================================

def compute_composite_beyer(df, weights=None):
    """
    Blend multiple Beyer estimates into a single composite.
    Uses confidence-adaptive weighting based on source disagreement.
    
    Default weights (no ML):  claude: 0.55, gemini: 0.45
    With ML:                  ml: 0.45, claude: 0.30, gemini: 0.25
    """
    beyer_cols = [c for c in df.columns if c.endswith('_beyer') and 
                  c not in ('composite_beyer', 'beyer_disagreement')]
    
    if not beyer_cols:
        raise ValueError("No Beyer columns found in DataFrame")
    
    if weights is None:
        if 'ml_beyer' in beyer_cols:
            weights = {'ml_beyer': 0.45, 'claude_beyer': 0.45, 'gemini_beyer': 0.1}
        elif len(beyer_cols) == 2:
            weights = {'claude_beyer': 0.9, 'gemini_beyer': 0.1}
        else:
            weights = {beyer_cols[0]: 1.0}
    
    # Only use available columns
    available = {k: v for k, v in weights.items() if k in df.columns}
    total_w = sum(available.values())
    available = {k: v / total_w for k, v in available.items()}
    
    result_df = df.copy()
    
    if len(available) >= 2:
        beyer_matrix = result_df[list(available.keys())].values
        row_std = np.nanstd(beyer_matrix, axis=1)
        result_df['beyer_disagreement'] = row_std
        
        LOW_THRESH = 5.0
        HIGH_THRESH = 15.0
        
        composites = np.zeros(len(result_df))
        for i in range(len(result_df)):
            if row_std[i] < LOW_THRESH:
                w = {k: 1.0 / len(available) for k in available}
            elif row_std[i] > HIGH_THRESH:
                if 'ml_beyer' in available:
                    w = {'ml_beyer': 0.45, 'claude_beyer': 0.45, 'gemini_beyer': 0.1}
                else:
                    w = {'claude_beyer': 0.9, 'gemini_beyer': 0.1}
                w = {k: v for k, v in w.items() if k in available}
            else:
                w = available.copy()
            
            tw = sum(w.values())
            vals = sum(result_df.iloc[i][col] * (w[col] / tw)
                       for col in w 
                       if col in result_df.columns and not pd.isna(result_df.iloc[i][col]))
            composites[i] = vals
        
        result_df['composite_beyer'] = composites
    else:
        col = list(available.keys())[0]
        result_df['composite_beyer'] = result_df[col]
        result_df['beyer_disagreement'] = 0.0
    
    return result_df


In [91]:
# =============================================================================
# 3. BEYER -> WIN PROBABILITY (t-distribution Monte Carlo)
# =============================================================================

def beyer_to_win_probabilities(df, composite_col='composite_beyer',
                                std_dev=4.8354, degrees_of_freedom=0.8108,
                                num_simulations=100000,
                                group_cols=None, seed=42):
    """
    Convert composite Beyer ratings to win probabilities using
    t-distribution Monte Carlo simulation.
    
    From backtest notebook Cell 7, optimized params from Cell 9:
        std_dev=4.8354, degrees_of_freedom=0.8108
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    # Use only columns that exist
    group_cols = [c for c in group_cols if c in df.columns]
    
    result_df = df.copy()
    result_df['projected_win_probability'] = 0.0
    
    rng = np.random.RandomState(seed)
    t_dist = stats.t(df=degrees_of_freedom)
    
    for group_vals, race_df in result_df.groupby(group_cols):
        race_indices = race_df.index
        num_horses = len(race_indices)
        
        if num_horses <= 1:
            result_df.loc[race_indices, 'projected_win_probability'] = 1.0
            continue
        
        composite_values = race_df[composite_col].values.reshape(num_horses, 1)
        random_effects = t_dist.rvs(size=(num_horses, num_simulations),
                                     random_state=rng) * std_dev
        finishes = composite_values + random_effects
        winners = np.argmax(finishes, axis=0)
        win_counts = np.bincount(winners, minlength=num_horses)
        win_probs = np.maximum(win_counts / num_simulations, 1e-6)
        win_probs = win_probs / win_probs.sum()
        
        result_df.loc[race_indices, 'projected_win_probability'] = win_probs
    
    return result_df


# =============================================================================
# 4. MORNING LINE PROCESSING
# =============================================================================

def parse_ml_odds(odds_str):
    """Parse fractional odds string to implied probability."""
    try:
        if pd.isna(odds_str) or str(odds_str).strip() == '':
            return np.nan
        odds_str = str(odds_str).strip()
        if '/' in odds_str:
            num, den = map(float, odds_str.split('/'))
            return den / (num + den)
        elif odds_str.upper() in ('EVN', 'EVEN', 'EV'):
            return 0.5
        else:
            decimal_odds = float(odds_str)
            if decimal_odds > 0:
                return 1 / (decimal_odds + 1)  # American-style: 5/1 stored as 5
            return np.nan
    except (ValueError, ZeroDivisionError):
        return np.nan


def process_morning_lines(df, ml_odds_col='ml_odds',
                          group_cols=None, temperature=1.0):
    """
    Convert morning line odds to normalized probabilities using
    softmax with temperature scaling.
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    group_cols = [c for c in group_cols if c in df.columns]
    
    result_df = df.copy()
    result_df['ml_probability_raw'] = result_df[ml_odds_col].apply(parse_ml_odds)
    result_df['ml_probability_normalized'] = np.nan
    
    for group_vals, group_df in result_df.groupby(group_cols):
        group_indices = group_df.index
        valid_mask = ~group_df['ml_probability_raw'].isna()
        valid_indices = group_indices[valid_mask]
        
        if len(valid_indices) > 0:
            valid_probs = group_df.loc[valid_indices, 'ml_probability_raw'].values
            logits = np.log(np.maximum(valid_probs, 1e-10)) / temperature
            exp_logits = np.exp(logits - np.max(logits))
            normalized = exp_logits / exp_logits.sum()
            result_df.loc[valid_indices, 'ml_probability_normalized'] = normalized
    
    return result_df


In [92]:






# =============================================================================
# 5. PARAMETER OPTIMIZATION (requires historical results)
# =============================================================================

def optimize_simulation_params(df, composite_col='composite_beyer',
                                group_cols=None,
                                initial_std=4.8, initial_df=0.8,
                                num_simulations=10000):
    """
    Optimize std_dev and degrees_of_freedom to minimize log loss.
    Requires 'won' column (1 = winner, 0 = loser).
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    
    if 'won' not in df.columns:
        raise ValueError("Need 'won' column for optimization")
    
    def objective(params):
        std_dev, deg_free = params
        result = beyer_to_win_probabilities(
            df, composite_col=composite_col,
            std_dev=std_dev, degrees_of_freedom=deg_free,
            num_simulations=num_simulations,
            group_cols=group_cols
        )
        probs = np.clip(result['projected_win_probability'].values, 1e-6, 1 - 1e-6)
        return log_loss(result['won'].values, probs)
    
    result = minimize(
        objective,
        x0=np.array([initial_std, initial_df]),
        method='Nelder-Mead',
        options={'maxiter': 250, 'disp': True}
    )
    
    optimal_std, optimal_df = result.x
    print(f"Optimal std_dev: {optimal_std:.4f}")
    print(f"Optimal degrees_of_freedom: {optimal_df:.4f}")
    print(f"Final log loss: {result.fun:.6f}")
    
    return optimal_std, optimal_df, result.fun


def optimize_blend_weights(df, group_cols=None,
                           std_dev=4.8354, degrees_of_freedom=0.8108,
                           num_simulations=10000):
    """
    Optimize Claude / Gemini / ML blend weights via log loss.
    Uses softmax parameterization to keep weights positive and sum to 1.
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    
    beyer_cols = [c for c in ['claude_beyer', 'gemini_beyer', 'ml_beyer']
                  if c in df.columns]
    
    if len(beyer_cols) < 2:
        raise ValueError(f"Need ≥2 Beyer sources, found: {beyer_cols}")
    
    def objective(raw_weights):
        weights = np.exp(raw_weights) / np.sum(np.exp(raw_weights))
        weight_dict = dict(zip(beyer_cols, weights))
        
        test_df = df.copy()
        test_df['composite_beyer'] = sum(
            test_df[col] * w for col, w in weight_dict.items()
        )
        
        result = beyer_to_win_probabilities(
            test_df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
            num_simulations=num_simulations, group_cols=group_cols
        )
        probs = np.clip(result['projected_win_probability'].values, 1e-6, 1 - 1e-6)
        return log_loss(result['won'].values, probs)
    
    result = minimize(objective, np.zeros(len(beyer_cols)),
                      method='Nelder-Mead', options={'maxiter': 200, 'disp': True})
    
    optimal_weights = np.exp(result.x) / np.sum(np.exp(result.x))
    weight_dict = dict(zip(beyer_cols, optimal_weights))
    
    print("\nOptimal blend weights:")
    for col, w in weight_dict.items():
        print(f"  {col}: {w:.4f}")
    print(f"Final log loss: {result.fun:.6f}")
    
    return weight_dict, result.fun


# =============================================================================
# 5b. FIT PARAMS TO MORNING LINE (no results needed)
# =============================================================================

def optimize_params_to_ml(df, composite_col='composite_beyer',
                           ml_prob_col='ml_probability_normalized',
                           group_cols=None,
                           initial_std=4.0, initial_df=2.0,
                           num_simulations=50000):
    """
    Optimize std_dev and degrees_of_freedom so that the model's 
    win probabilities best match ML-implied probabilities.
    
    No results needed — uses KL divergence from ML as the objective.
    This calibrates your Beyer gaps to produce the same probability 
    spread the oddsmaker sees.
    
    Parameters
    ----------
    df : DataFrame with composite_beyer and ml_probability_normalized
    
    Returns
    -------
    (optimal_std, optimal_df, final_kl)
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    
    if ml_prob_col not in df.columns:
        raise ValueError(f"Need '{ml_prob_col}' column. Run process_morning_lines first.")
    
    # Filter to rows with valid ML probs
    valid = df[df[ml_prob_col].notna() & (df[ml_prob_col] > 0)].copy()
    
    if len(valid) == 0:
        raise ValueError("No valid ML probabilities found")
    
    ml_target = valid[ml_prob_col].values
    
    def objective(params):
        std_dev, deg_free = params
        result = beyer_to_win_probabilities(
            valid, composite_col=composite_col,
            std_dev=std_dev, degrees_of_freedom=deg_free,
            num_simulations=num_simulations,
            group_cols=group_cols
        )
        model_probs = np.clip(result['projected_win_probability'].values, 1e-8, 1 - 1e-8)
        ml_probs = np.clip(ml_target, 1e-8, 1 - 1e-8)
        
        # KL divergence: sum(p * log(p / q))
        # where p = ML (the "truth" we're fitting to), q = model
        kl = np.sum(ml_probs * np.log(ml_probs / model_probs))
        return kl
    
    print(f"Optimizing std/df to match ML probabilities ({len(valid)} horses)...")
    
    result = minimize(
        objective,
        x0=np.array([initial_std, initial_df]),
        method='Nelder-Mead',
        options={'maxiter': 150, 'disp': True, 'xatol': 0.05, 'fatol': 0.001}
    )
    
    optimal_std, optimal_df = result.x
    print(f"\nOptimal std_dev: {optimal_std:.4f}")
    print(f"Optimal degrees_of_freedom: {optimal_df:.4f}")
    print(f"Final KL divergence: {result.fun:.6f}")
    
    # Show calibration comparison
    final = beyer_to_win_probabilities(
        valid, composite_col=composite_col,
        std_dev=optimal_std, degrees_of_freedom=optimal_df,
        num_simulations=num_simulations * 2,
        group_cols=group_cols
    )
    
    comp = pd.DataFrame({
        'ml': valid[ml_prob_col].values,
        'model': final['projected_win_probability'].values
    })
    comp['bucket'] = pd.cut(comp['ml'], bins=[0, 0.05, 0.10, 0.20, 0.35, 1.0],
                             labels=['<5%', '5-10%', '10-20%', '20-35%', '>35%'])
    
    cal = comp.groupby('bucket').agg(
        n=('ml', 'count'),
        avg_ml=('ml', 'mean'),
        avg_model=('model', 'mean')
    )
    print("\nCalibration (ML vs Model):")
    print(cal)
    
    return optimal_std, optimal_df, result.fun


# =============================================================================
# 6. EDGE CALCULATION
# =============================================================================

def calculate_edge(df, projected_prob_col='projected_win_probability',
                   ml_prob_col='ml_probability_normalized'):
    """Calculate edge, Kelly, expected ROI between model and ML."""
    result_df = df.copy()
    
    ml = result_df[ml_prob_col]
    proj = result_df[projected_prob_col]
    
    result_df['edge'] = proj - ml
    result_df['percentage_edge'] = np.where(ml > 0, ((proj / ml) - 1) * 100, np.nan)
    
    result_df['kelly_stake'] = np.where(
        ml > 0,
        np.clip((proj * (1 / ml) - 1) / (1 / ml - 1), 0, 1),
        np.nan
    )
    result_df['half_kelly'] = result_df['kelly_stake'] / 2
    
    result_df['expected_roi'] = np.where(ml > 0, (proj / ml) - 1, np.nan)
    
    return result_df





In [93]:
def run_backtest(base_dir='../data/ai_picks', entries_dir='../data/entries',
                 std_dev=5.81, degrees_of_freedom=1.171,
                 optimize=False, num_simulations=50000):
    """
    Backtest across all available dates with results.
    Only considers dates that have AI picks.
    Prints log loss for ML, Claude, Gemini, and composite.
    """
    group_cols = ['date', 'track', 'race']
    
    picks_df = load_all_ai_dates(base_dir)
    if picks_df.empty:
        print("No AI picks found.")
        return None
    
    # Normalize AI pick dates from folder names like '4_3_26' to '2026-04-03'
    def normalize_date(d):
        """Convert various date formats to YYYY-MM-DD."""
        d = str(d).strip()
        if '_' in d:
            parts = d.split('_')
            if len(parts) == 3:
                m, day, y = parts
                y = f"20{y}" if len(y) == 2 else y
                return f"{y}-{int(m):02d}-{int(day):02d}"
        # Already YYYY-MM-DD or parseable
        try:
            return pd.to_datetime(d).strftime('%Y-%m-%d')
        except:
            return d
    
    picks_df['date'] = picks_df['date'].apply(normalize_date)
    ai_dates = picks_df['date'].unique()
    print(f"Found AI picks for {len(ai_dates)} date(s): {', '.join(sorted(ai_dates))}")
    
    # Load entries
    entries = load_entries(entries_dir)
    if entries.empty:
        print(f"No results data found in {entries_dir}.")
        return None
    
    print(f"Entries loaded: {len(entries)} rows, dates: {sorted(entries['date'].unique())}")
    
    # Filter entries to only dates with AI picks
    entries = entries[entries['date'].isin(ai_dates)].copy()
    if entries.empty:
        all_entry_dates = load_entries(entries_dir)['date'].unique() if not load_entries(entries_dir).empty else []
        print(f"No results found for AI pick dates {sorted(ai_dates)}.")
        print(f"Available result dates: {sorted(all_entry_dates)[:10]}")
        return None
    
    # Check for finish/won data
    if 'won' not in entries.columns:
        print("No 'finish' column in results — results may not have been scraped yet.")
        return None
    
    # Merge picks with entries
    df = picks_df.merge(entries, on=['date', 'track', 'race', 'horse_number'], how='inner')
    
    if df.empty:
        print("No matching picks+results found after merge.")
        print(f"  Picks columns: {list(picks_df.columns)}")
        print(f"  Entries columns: {list(entries.columns)}")
        print(f"  Sample picks horse_numbers: {picks_df['horse_number'].head().tolist()}")
        print(f"  Sample entries horse_numbers: {entries['horse_number'].head().tolist()}")
        return None
    
    print(f"\nBacktest: {len(df)} entries, {df['date'].nunique()} dates, "
          f"{df.groupby(group_cols).ngroups} races")
    
    # ---- ML log loss (baseline) ----
    if 'ml_odds' in df.columns:
        df = process_morning_lines(df, ml_odds_col='ml_odds', group_cols=group_cols)
        if 'ml_probability_normalized' in df.columns:
            ml_valid = df['ml_probability_normalized'].notna()
            if ml_valid.sum() > 0:
                ml_probs = np.clip(df.loc[ml_valid, 'ml_probability_normalized'].values, 1e-6, 1-1e-6)
                ml_loss = log_loss(df.loc[ml_valid, 'won'].values, ml_probs)
                print(f"\n--- LOG LOSS COMPARISON ---")
                print(f"  Morning Line:  {ml_loss:.6f}")
    
    # ---- Per-source log loss (before blending) ----
    beyer_cols = [c for c in df.columns if c.endswith('_beyer')
                  and c not in ('composite_beyer', 'beyer_disagreement')]
    
    for bcol in beyer_cols:
        source_name = bcol.replace('_beyer', '').replace('_expected', '')
        try:
            source_df = beyer_to_win_probabilities(
                df, composite_col=bcol,
                std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
                num_simulations=num_simulations // 5, group_cols=group_cols
            )
            source_probs = np.clip(source_df['projected_win_probability'].values, 1e-6, 1-1e-6)
            source_loss = log_loss(source_df['won'].values, source_probs)
            print(f"  {source_name:>14}:  {source_loss:.6f}")
        except Exception as e:
            print(f"  {source_name:>14}:  ERROR - {e}")
    
    # ---- Composite ----
    df = compute_composite_beyer(df)
    
    if optimize:
        print("\nOptimizing simulation parameters...")
        std_dev, degrees_of_freedom, _ = optimize_simulation_params(
            df, num_simulations=num_simulations // 5, group_cols=group_cols
        )
        
        if len(beyer_cols) >= 2:
            print("\nOptimizing blend weights...")
            weights, _ = optimize_blend_weights(
                df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
                num_simulations=num_simulations // 5, group_cols=group_cols
            )
    
    # Final run
    df = beyer_to_win_probabilities(
        df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
        num_simulations=num_simulations, group_cols=group_cols
    )
    
    # Evaluate composite
    probs = np.clip(df['projected_win_probability'].values, 1e-6, 1 - 1e-6)
    loss = log_loss(df['won'].values, probs)
    print(f"  {'Composite':>14}:  {loss:.6f}")
    print(f"{'---':>30}")
    
    # ---- Beyer MSE/MAE (if actual_beyer column exists) ----
    if 'actual_beyer' in df.columns:
        beyer_eval = evaluate_beyer_accuracy(df, beyer_cols)
    
    # ---- Ranking accuracy (uses all horses, no selection bias) ----
    if 'finish' in df.columns:
        ranking_eval = evaluate_ranking_accuracy(df, beyer_cols)
    
    # Calibration
    df['prob_bucket'] = pd.cut(
        df['projected_win_probability'],
        bins=[0, 0.05, 0.10, 0.20, 0.35, 1.0],
        labels=['<5%', '5-10%', '10-20%', '20-35%', '>35%']
    )
    cal = df.groupby('prob_bucket').agg(
        n=('won', 'count'),
        actual=('won', 'mean'),
        predicted=('projected_win_probability', 'mean')
    )
    print("\nCalibration:")
    print(cal)
    
    return df
 
 
def evaluate_beyer_accuracy(df, beyer_cols=None):
    """
    Evaluate how accurate each Beyer source is against actual Beyers.
    
    Prints MSE, RMSE, MAE, and bias for each source + composite.
    Also fits a simple linear regression to check calibration slope.
    
    Parameters
    ----------
    df : DataFrame
        Must contain 'actual_beyer' column and one or more *_beyer columns.
    beyer_cols : list, optional
        Which Beyer estimate columns to evaluate. Auto-detected if None.
    
    Returns
    -------
    DataFrame with accuracy metrics per source.
    """
    if 'actual_beyer' not in df.columns:
        print("No actual_beyer column found — skipping Beyer accuracy eval.")
        return None
    
    # Filter to rows with actual beyers
    mask = df['actual_beyer'].notna()
    if mask.sum() == 0:
        print("No actual Beyer data available yet.")
        return None
    
    eval_df = df[mask].copy()
    n = len(eval_df)
    
    if beyer_cols is None:
        beyer_cols = [c for c in eval_df.columns if c.endswith('_beyer')
                      and c not in ('composite_beyer', 'beyer_disagreement', 'actual_beyer')]
    
    # Always include composite if present
    all_cols = beyer_cols.copy()
    if 'composite_beyer' in eval_df.columns:
        all_cols.append('composite_beyer')
    
    print(f"\n--- BEYER ACCURACY ({n} horses with actual Beyers) ---")
    print(f"  {'Source':>14}  {'MSE':>7}  {'RMSE':>6}  {'MAE':>6}  {'Bias':>6}  {'Corr':>6}  {'Slope':>6}")
    
    results = []
    for col in all_cols:
        if col not in eval_df.columns:
            continue
        
        valid = eval_df[col].notna()
        if valid.sum() < 5:
            continue
        
        predicted = eval_df.loc[valid, col].values
        actual = eval_df.loc[valid, 'actual_beyer'].values
        
        residuals = predicted - actual
        mse = np.mean(residuals ** 2)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(residuals))
        bias = np.mean(residuals)  # positive = overestimates
        corr = np.corrcoef(predicted, actual)[0, 1]
        
        # Simple linear regression: actual = a + b * predicted
        # slope < 1 means estimates are too spread, > 1 means too compressed
        if np.std(predicted) > 0:
            slope = np.cov(actual, predicted)[0, 1] / np.var(predicted)
        else:
            slope = np.nan
        
        label = col.replace('_beyer', '').replace('_expected', '')
        print(f"  {label:>14}  {mse:>7.1f}  {rmse:>6.1f}  {mae:>6.1f}  {bias:>+6.1f}  {corr:>6.3f}  {slope:>6.3f}")
        
        results.append({
            'source': label,
            'n': valid.sum(),
            'mse': mse, 'rmse': rmse, 'mae': mae,
            'bias': bias, 'corr': corr, 'slope': slope
        })
    
    print(f"{'':>14}  (Bias: + = overestimates, Slope: 1.0 = perfect calibration)")
    
    results_df = pd.DataFrame(results)
    
    # Per-race analysis: which source won?
    if len(beyer_cols) >= 2 and len(beyer_cols) <= 3:
        print(f"\n  Per-race MSE wins:")
        for col in beyer_cols:
            label = col.replace('_beyer', '')
            valid = eval_df.groupby(['date', 'track', 'race']).apply(
                lambda g: np.mean((g[col] - g['actual_beyer']) ** 2) 
                if g[col].notna().sum() > 0 and g['actual_beyer'].notna().sum() > 0
                else np.nan
            ).dropna()
            
            other_cols = [c for c in beyer_cols if c != col]
            if other_cols:
                other_valid = eval_df.groupby(['date', 'track', 'race']).apply(
                    lambda g: np.mean((g[other_cols[0]] - g['actual_beyer']) ** 2)
                    if g[other_cols[0]].notna().sum() > 0 and g['actual_beyer'].notna().sum() > 0
                    else np.nan
                ).dropna()
                
                common = valid.index.intersection(other_valid.index)
                if len(common) > 0:
                    wins = (valid[common] < other_valid[common]).sum()
                    total = len(common)
                    print(f"    {label} beat {other_cols[0].replace('_beyer','')}: "
                          f"{wins}/{total} races ({wins/total:.0%})")
    
    return results_df
 
 
def evaluate_ranking_accuracy(df, beyer_cols=None):
    """
    Evaluate how well each Beyer source ranks horses by finish position.
    Uses ALL horses with finish data (no selection bias).
    
    Metrics:
    - Within-race Spearman correlation (estimated Beyer vs finish)
    - Top-pick hit rate (did the source's top pick finish 1st, 2nd, 3rd?)
    - Winner identification rate
    
    Parameters
    ----------
    df : DataFrame with finish column and *_beyer columns
    
    Returns
    -------
    DataFrame with ranking metrics per source.
    """
    if 'finish' not in df.columns:
        print("No finish data — skipping ranking evaluation.")
        return None
    
    group_cols = ['date', 'track', 'race']
    group_cols = [c for c in group_cols if c in df.columns]
    
    if beyer_cols is None:
        beyer_cols = [c for c in df.columns if c.endswith('_beyer')
                      and c not in ('composite_beyer', 'beyer_disagreement', 'actual_beyer', 'prior_beyer')]
    
    all_cols = beyer_cols.copy()
    if 'composite_beyer' in df.columns:
        all_cols.append('composite_beyer')
    
    # Filter to running horses with finish data
    eval_df = df[df['finish'].notna()].copy()
    n_races = eval_df.groupby(group_cols).ngroups
    n_horses = len(eval_df)
    
    print(f"\n--- RANKING ACCURACY ({n_horses} horses, {n_races} races) ---")
    print(f"  {'Source':>14}  {'SpearmanR':>9}  {'TopPick%':>8}  {'Top3Hit%':>8}  {'WinnerRk':>8}")
    
    results = []
    for col in all_cols:
        if col not in eval_df.columns:
            continue
        
        spearman_corrs = []
        top_pick_wins = 0
        top_pick_top3 = 0
        winner_ranks = []
        n_valid_races = 0
        
        for group_vals, race_df in eval_df.groupby(group_cols):
            valid = race_df[col].notna() & race_df['finish'].notna()
            race_valid = race_df[valid]
            
            if len(race_valid) < 3:
                continue
            
            n_valid_races += 1
            
            # Spearman: higher beyer should = lower finish (1st)
            # So we expect negative correlation (high beyer -> low finish number)
            from scipy.stats import spearmanr
            corr, _ = spearmanr(race_valid[col].values, race_valid['finish'].values)
            spearman_corrs.append(corr)
            
            # Top pick: horse with highest estimated beyer
            top_pick_idx = race_valid[col].idxmax()
            top_pick_finish = race_valid.loc[top_pick_idx, 'finish']
            if top_pick_finish == 1:
                top_pick_wins += 1
            if top_pick_finish <= 3:
                top_pick_top3 += 1
            
            # Winner rank: where did the actual winner rank in our estimates?
            winner_mask = race_valid['finish'] == 1
            if winner_mask.sum() > 0:
                winner_beyer = race_valid.loc[winner_mask, col].values[0]
                # Rank = how many horses had higher estimated beyer + 1
                rank = (race_valid[col] > winner_beyer).sum() + 1
                winner_ranks.append(rank)
        
        if n_valid_races == 0:
            continue
        
        avg_spearman = np.mean(spearman_corrs)
        top_pick_pct = top_pick_wins / n_valid_races * 100
        top3_pct = top_pick_top3 / n_valid_races * 100
        avg_winner_rank = np.mean(winner_ranks)
        
        label = col.replace('_beyer', '').replace('_expected', '')
        print(f"  {label:>14}  {avg_spearman:>+9.3f}  {top_pick_pct:>7.1f}%  {top3_pct:>7.1f}%  {avg_winner_rank:>8.1f}")
        
        results.append({
            'source': label,
            'n_races': n_valid_races,
            'spearman': avg_spearman,
            'top_pick_win_pct': top_pick_pct,
            'top_pick_top3_pct': top3_pct,
            'avg_winner_rank': avg_winner_rank,
        })
    
    print(f"  {'':>14}  (Spearman: -1.0=perfect, TopPick%: source's #1 won, WinnerRk: 1.0=perfect)")
    
    # ML comparison if available
    if 'ml_probability_normalized' in eval_df.columns:
        ml_top_wins = 0
        ml_top3 = 0
        ml_winner_ranks = []
        n_ml_races = 0
        ml_spearman_corrs = []
        
        for group_vals, race_df in eval_df.groupby(group_cols):
            valid = race_df['ml_probability_normalized'].notna() & race_df['finish'].notna()
            race_valid = race_df[valid]
            if len(race_valid) < 3:
                continue
            
            n_ml_races += 1
            corr, _ = spearmanr(race_valid['ml_probability_normalized'].values, 
                               race_valid['finish'].values)
            ml_spearman_corrs.append(corr)
            
            fav_idx = race_valid['ml_probability_normalized'].idxmax()
            fav_finish = race_valid.loc[fav_idx, 'finish']
            if fav_finish == 1:
                ml_top_wins += 1
            if fav_finish <= 3:
                ml_top3 += 1
            
            winner_mask = race_valid['finish'] == 1
            if winner_mask.sum() > 0:
                winner_prob = race_valid.loc[winner_mask, 'ml_probability_normalized'].values[0]
                rank = (race_valid['ml_probability_normalized'] > winner_prob).sum() + 1
                ml_winner_ranks.append(rank)
        
        if n_ml_races > 0:
            print(f"  {'morning_line':>14}  {np.mean(ml_spearman_corrs):>+9.3f}  "
                  f"{ml_top_wins/n_ml_races*100:>7.1f}%  {ml_top3/n_ml_races*100:>7.1f}%  "
                  f"{np.mean(ml_winner_ranks):>8.1f}")
    
    return pd.DataFrame(results) if results else None
 
 

In [94]:
# =============================================================================
# 7. FULL PIPELINE
# =============================================================================

def run_pipeline(date_str, track='keeneland',
                 base_dir='../data/ai_picks',
                 entries_dir='../data/entries',
                 upc_dir='../data/upc_entries',
                 std_dev=4.8354, degrees_of_freedom=0.8108,
                 blend_weights=None, ml_temperature=1.0,
                 num_simulations=100000):
    """
    Full pipeline for a single race date:
    1. Load AI picks (Claude, Gemini)
    2. Compute composite Beyer
    3. Convert to win probabilities via MC simulation
    4. Load ML odds from upc_entries or entries
    5. Calculate edges
    6. Merge results if available
    """
    group_cols = ['date', 'track', 'race']
    
    # --- Step 1: Load AI picks ---
    print(f"Loading AI picks for {date_str}...")
    df = load_ai_picks(date_str, base_dir)
    
    # Ensure track column
    if 'track' not in df.columns:
        df['track'] = track
    
    print(f"  {len(df)} entries across {df['race'].nunique()} races")
    
    # --- Step 2: Load ML / filter scratches (before simulation) ---
    ml_merged = False
    
    upc = load_upc_entries(track, upc_dir)
    if upc is not None and 'ml_odds' in upc.columns:
        print(f"Loading ML odds from upc_entries/{track}.csv...")
        if 'horse' in upc.columns and 'horse_number' not in upc.columns:
            upc['horse_number'] = upc['horse'].astype(str).str.strip()
        
        # Filter to running horses only (scratch filter)
        if 'running' in upc.columns:
            scratches_before = len(df)
            running_set = set(
                zip(upc.loc[upc['running'] == 1, 'race'],
                    upc.loc[upc['running'] == 1, 'horse_number'])
            )
            df = df[df.apply(lambda r: (r['race'], r['horse_number']) in running_set, axis=1)].copy()
            n_scratched = scratches_before - len(df)
            if n_scratched > 0:
                print(f"  Removed {n_scratched} scratches, {len(df)} runners remain")
        
        merge_cols = ['race', 'horse_number']
        upc_keep = ['race', 'horse_number', 'ml_odds']
        if 'horse_name' in upc.columns:
            upc_keep.append('horse_name')
        upc_subset = upc[[c for c in upc_keep if c in upc.columns]].drop_duplicates()
        df = df.merge(upc_subset, on=merge_cols, how='left')
        ml_merged = True
    
    if not ml_merged:
        # Try entries dir
        date_fmt = pd.to_datetime(date_str.replace('_', '-')).strftime('%Y-%m-%d') \
            if '_' in date_str else date_str
        entries = load_entries(entries_dir, track=track, date=date_fmt)
        if not entries.empty and 'ml_odds' in entries.columns:
            print(f"Loading ML odds from entries...")
            # entries already filtered to running==1 in load_entries
            scratches_before = len(df)
            running_set = set(zip(entries['race'], entries['horse_number']))
            df = df[df.apply(lambda r: (r['race'], r['horse_number']) in running_set, axis=1)].copy()
            n_scratched = scratches_before - len(df)
            if n_scratched > 0:
                print(f"  Removed {n_scratched} scratches, {len(df)} runners remain")
            
            entries_keep = ['race', 'horse_number', 'ml_odds']
            if 'horse_name' in entries.columns:
                entries_keep.append('horse_name')
            entries_subset = entries[[c for c in entries_keep if c in entries.columns]].drop_duplicates()
            df = df.merge(entries_subset, on=['race', 'horse_number'], how='left')
            ml_merged = True
    
    if ml_merged and 'ml_odds' in df.columns:
        df = process_morning_lines(df, ml_odds_col='ml_odds',
                                    group_cols=group_cols,
                                    temperature=ml_temperature)
    
    # --- Step 3: Composite Beyer (after scratches removed) ---
    print("Computing composite Beyer estimates...")
    df = compute_composite_beyer(df, weights=blend_weights)
    
    # --- Step 4: Win probabilities (on actual runners only) ---
    print(f"Running MC simulation (n={num_simulations:,}, "
          f"std={std_dev:.4f}, df={degrees_of_freedom:.4f})...")
    df = beyer_to_win_probabilities(
        df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
        num_simulations=num_simulations, group_cols=group_cols
    )
    
    # --- Step 5: Edges ---
    if 'ml_probability_normalized' in df.columns:
        print("Calculating edges...")
        df = calculate_edge(df)
    
    # --- Step 6: Results ---
    date_fmt = date_str
    if '_' in date_str:
        # Convert 4_3_26 -> 2026-04-03
        parts = date_str.split('_')
        if len(parts) == 3:
            m, d, y = parts
            y = f"20{y}" if len(y) == 2 else y
            date_fmt = f"{y}-{int(m):02d}-{int(d):02d}"
    
    entries = load_entries(entries_dir, track=track, date=date_fmt)
    if not entries.empty:
        print("Merging results for backtesting...")
        result_cols = ['race', 'horse_number', 'finish', 'won',
                       'win_paid', 'place_paid', 'show_paid', 'payout_per_dollar']
        result_cols = [c for c in result_cols if c in entries.columns]
        
        # Avoid duplicate ml_odds
        if 'ml_odds' in entries.columns and 'ml_odds' in df.columns:
            result_cols = [c for c in result_cols if c != 'ml_odds']
        
        df = df.merge(entries[result_cols].drop_duplicates(),
                      on=['race', 'horse_number'], how='left')
    
    # --- Summary ---
    beyer_cols = [c for c in df.columns if c.endswith('_beyer')
                  and c not in ('composite_beyer', 'beyer_disagreement')]
    print(f"\nPipeline complete: {len(df)} horses, {df['race'].nunique()} races")
    print(f"  Sources: {beyer_cols}")
    print(f"  ML: {'ml_probability_normalized' in df.columns}")
    print(f"  Results: {'won' in df.columns}")
    
    return df

In [95]:
# =============================================================================
# 8. REPORTING
# =============================================================================

def print_race_report(df, race_num):
    """Formatted report for a single race."""
    race = df[df['race'] == race_num].sort_values('projected_win_probability', ascending=False)
    
    print(f"\n{'='*72}")
    print(f"RACE {race_num}")
    print(f"{'='*72}")
    
    beyer_cols = [c for c in race.columns if c.endswith('_beyer')
                  and c not in ('composite_beyer', 'beyer_disagreement')]
    
    header = f"{'#':>3} {'Comp':>5} "
    for col in beyer_cols:
        header += f"{col.replace('_beyer','')[:6]:>6} "
    header += f"{'WinP':>7} "
    
    if 'ml_probability_normalized' in race.columns:
        header += f"{'ML%':>6} {'Edge%':>7} {'Kelly':>6}"
    if 'beyer_disagreement' in race.columns:
        header += f" {'Disg':>5}"
    
    print(header)
    print("-" * len(header))
    
    for _, row in race.iterrows():
        line = f"{str(row['horse_number']):>3} {row['composite_beyer']:>5.1f} "
        for col in beyer_cols:
            v = row.get(col, np.nan)
            line += f"{v:>6.1f} " if not pd.isna(v) else f"{'—':>6} "
        line += f"{row['projected_win_probability']:>6.1%} "
        
        if 'ml_probability_normalized' in race.columns:
            ml = row.get('ml_probability_normalized', np.nan)
            edge = row.get('percentage_edge', np.nan)
            kelly = row.get('kelly_stake', np.nan)
            line += f"{ml:>5.1%} " if not pd.isna(ml) else f"{'—':>5} "
            line += f"{edge:>+6.1f}% " if not pd.isna(edge) else f"{'—':>7} "
            line += f"{kelly:>5.3f}" if not pd.isna(kelly) else f"{'—':>5}"
        if 'beyer_disagreement' in race.columns:
            line += f" {row['beyer_disagreement']:>5.1f}"
        
        print(line)
 
 
def percentage_to_american_odds(probability):
    """Convert win probability to American odds."""
    if probability <= 0 or probability >= 1:
        return None
    if probability >= 0.5:
        return int(-100 * probability / (1 - probability))
    else:
        return int(100 * (1 - probability) / probability)

In [96]:
# Uncomment when results exist:
run_backtest(optimize=True)

Found AI picks for 2 date(s): 2026-04-03, 2026-04-04
Entries loaded: 4019 rows, dates: ['2023-04-21', '2023-04-22', '2023-04-23', '2023-04-26', '2023-04-27', '2023-04-28', '2023-04-29', '2023-05-02', '2023-05-03', '2023-05-04', '2023-05-05', '2023-05-06', '2023-05-11', '2023-05-12', '2023-05-13', '2023-05-14', '2023-05-18', '2023-05-19', '2023-05-20', '2023-11-03', '2023-11-04', '2024-01-27', '2025-04-18', '2025-04-19', '2025-04-24', '2025-04-25', '2025-04-26', '2025-04-30', '2025-05-01', '2025-05-02', '2025-05-15', '2025-05-16', '2025-05-17', '2025-05-22', '2025-05-23', '2025-05-26', '2025-05-29', '2025-05-30', '2025-05-31', '2025-06-06', '2026-04-03', '2026-04-04']

Backtest: 221 entries, 2 dates, 21 races

--- LOG LOSS COMPARISON ---
  Morning Line:  0.274606
          claude:  0.288116
          gemini:  0.379990
           prior:  0.807863
          actual:  2.417070

Optimizing simulation parameters...
Optimization terminated successfully.
         Current function value: 0.28858

C:\Users\Blake\AppData\Local\Temp\ipykernel_6864\1985392973.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cal = df.groupby('prob_bucket').agg(


,race,horse_number,claude_beyer,gemini_beyer,date,track,running,horse,horse_name,sire,...,won,place,show,payout_per_dollar,ml_probability_raw,ml_probability_normalized,composite_beyer,beyer_disagreement,projected_win_probability,prob_bucket
0,1,1,45.0,56.0,2026-04-03,keeneland,1,1,Fortune King,Catalina Cruiser,...,0,0,0,0.00,0.062500,0.049561,45.0,0.0,0.05256,5-10%
1,1,2,58.0,62.0,2026-04-03,keeneland,1,2,Joe Joe Dude,Practical Joke,...,0,0,0,0.00,0.200000,0.158594,58.0,0.0,0.10586,10-20%
2,1,3,42.0,52.0,2026-04-03,keeneland,1,3,Super Saiyajin,Modernist,...,0,0,0,0.00,0.076923,0.060998,42.0,0.0,0.04936,<5%
3,1,4,38.0,48.0,2026-04-03,keeneland,1,4,Autism Honor G,Highly Motivated,...,0,0,0,0.00,0.047619,0.037760,38.0,0.0,0.04196,<5%
4,1,5,55.0,60.0,2026-04-03,keeneland,1,5,Bourbon Town,Echo Town,...,0,1,1,0.00,0.142857,0.113281,55.0,0.0,0.08544,5-10%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,11,5,78.0,85.0,2026-04-04,keeneland,1,5,Creole Chrome,Volatile,...,0,0,0,0.00,0.111111,0.087544,78.0,0.0,0.07532,5-10%
217,11,6,90.0,81.0,2026-04-04,keeneland,1,6,Further Ado,Gun Runner,...,1,1,1,1.85,0.384615,0.303036,90.0,0.0,0.24074,20-35%
218,11,7,72.0,70.0,2026-04-04,keeneland,1,7,Ocelli,Connect,...,0,0,0,0.00,0.032258,0.025416,72.0,0.0,0.05740,5-10%
219,11,8,68.0,75.0,2026-04-04,keeneland,1,8,Moonstrocity,Tiz the Law,...,0,0,0,0.00,0.032258,0.025416,68.0,0.0,0.04872,<5%


In [82]:
# opt_std, opt_df, kl = optimize_params_to_ml(df)

In [84]:
"""
pick5_optimizer.py

Optimize Pick 4/5/6 tickets using AI ensemble win probabilities.
Uses the output from ai_picks_pipeline.py.

Without ML odds available, uses projected_win_probability directly
as both the "true" probability and the market proxy (conservative approach).
When ML odds are available, blends model + ML and uses ML as market for EV calc.
"""

import numpy as np
import pandas as pd
from operator import itemgetter
# from ai_picks_pipeline import (
#     load_ai_picks, compute_composite_beyer, beyer_to_win_probabilities,
#     process_morning_lines, print_race_report, run_pipeline
# )


# =============================================================================
# PICK TICKET OPTIMIZER
# =============================================================================

def optimize_pick_tickets(df, race_numbers, bet_type=None, blend_factor=0.5,
                          pool_size=100000, takeout_rate=0.22,
                          max_cost=None, min_prob=None, min_total_prob=None,
                          budget_steps=20,
                          projected_prob_col='projected_win_probability',
                          ml_prob_col=None,
                          race_col='race', horse_col='horse_number',
                          debug=False):
    """
    Optimize Pick N tickets using model probabilities (optionally blended with ML).
    
    Parameters
    ----------
    df : DataFrame with race data
    race_numbers : list of race numbers to include
    bet_type : int, auto-detected from len(race_numbers) if None
    blend_factor : float, weight on model (1.0 = all model, 0.0 = all ML)
    pool_size : estimated pool size in dollars
    takeout_rate : track takeout (0.22 = 22%)
    max_cost : max ticket cost in combinations
    min_prob : min coverage per race (e.g. 0.7 = 70%)
    min_total_prob : min total hit probability
    budget_steps : number of frontier points to return
    
    Returns
    -------
    list of (cost, ev, probability, selections_dict)
    """
    if bet_type is None:
        bet_type = len(race_numbers)
    
    # Filter to requested races
    df_pick = df[df[race_col].isin(race_numbers)].copy()
    
    # Determine if we have ML probabilities
    has_ml = (ml_prob_col is not None and 
              ml_prob_col in df_pick.columns and 
              df_pick[ml_prob_col].notna().any())
    
    if not has_ml:
        # No ML: use projected prob as both model and "market"
        # This means EV calculation is neutral — we're just optimizing coverage
        blend_factor = 1.0
        ml_prob_col_used = projected_prob_col
        print("No ML odds available. Using model probabilities for ticket construction.")
        print("EV calculations will assume fair market (no overlay detection).")
    else:
        ml_prob_col_used = ml_prob_col
    
    # Calculate blended probabilities
    df_pick['blended_prob'] = (blend_factor * df_pick[projected_prob_col] + 
                                (1 - blend_factor) * df_pick[ml_prob_col_used])
    
    # Normalize within each race
    for race_num in race_numbers:
        mask = df_pick[race_col] == race_num
        total = df_pick.loc[mask, 'blended_prob'].sum()
        if total > 0:
            df_pick.loc[mask, 'blended_prob'] /= total
    
    # Build probability dicts
    races = sorted(race_numbers)
    blended_probs = {}
    market_probs = {}
    sorted_horses = {}
    
    for race_num in races:
        rkey = f"Race {race_num}"
        race_data = df_pick[df_pick[race_col] == race_num]
        
        blended_probs[rkey] = dict(zip(race_data[horse_col], race_data['blended_prob']))
        market_probs[rkey] = dict(zip(race_data[horse_col], race_data[ml_prob_col_used]))
        sorted_horses[rkey] = sorted(blended_probs[rkey].items(), 
                                      key=itemgetter(1), reverse=True)
    
    # Minimum selections per race
    min_selection = [1] * bet_type
    if min_prob is not None:
        for i, race in enumerate(sorted(blended_probs.keys())):
            cum = 0
            count = 0
            for _, prob in sorted_horses[race]:
                cum += prob
                count += 1
                if cum >= min_prob:
                    break
            min_selection[i] = max(1, count)
    
    max_horses = [len(sorted_horses[r]) for r in sorted(blended_probs.keys())]
    
    # Generate all allocations
    all_allocations = []
    for total in range(sum(min_selection), sum(max_horses) + 1):
        base = min_selection.copy()
        remaining = total - sum(base)
        
        def distribute(current, rem, idx=0):
            if idx == len(current) - 1:
                if current[idx] + rem <= max_horses[idx]:
                    alloc = current.copy()
                    alloc[idx] += rem
                    all_allocations.append(alloc)
                return
            for i in range(rem + 1):
                if current[idx] + i <= max_horses[idx]:
                    c = current.copy()
                    c[idx] += i
                    distribute(c, rem - i, idx + 1)
        
        distribute(base, remaining)
    
    # Evaluate each allocation
    results = []
    race_list = sorted(blended_probs.keys())
    net_pool = pool_size * (1 - takeout_rate)
    
    for allocation in all_allocations:
        cost = int(np.prod(allocation))
        
        if max_cost is not None and cost > max_cost:
            continue
        
        total_prob = 1.0
        raw_market_prob = 1.0
        selections = {}
        
        for i, race in enumerate(race_list):
            top = sorted_horses[race][:allocation[i]]
            race_prob = sum(p for _, p in top)
            market_p = sum(market_probs[race].get(h, 0) for h, _ in top)
            
            total_prob *= race_prob
            raw_market_prob *= market_p
            selections[race] = [h for h, _ in top]
        
        if min_total_prob is not None and total_prob < min_total_prob:
            continue
        
        if raw_market_prob > 0:
            expected_payout = (total_prob / raw_market_prob) * net_pool
            ev = expected_payout - cost
        else:
            ev = -cost
        
        results.append((cost, ev, total_prob, selections))
    
    # Sort by EV descending
    results.sort(key=lambda x: x[1], reverse=True)
    
    # Efficient frontier: best EV per cost level
    frontier = []
    best_ev = {}
    for r in results:
        cost = r[0]
        if cost not in best_ev or r[1] > best_ev[cost]:
            best_ev[cost] = r[1]
            frontier.append(r)
    
    frontier.sort(key=lambda x: x[0])
    
    if debug:
        print(f"\nEvaluated {len(all_allocations)} allocations, {len(results)} valid")
        print(f"Frontier: {len(frontier)} points")
    
    # Sample frontier points
    if frontier and budget_steps > 0 and budget_steps < len(frontier):
        indices = np.linspace(0, len(frontier) - 1, budget_steps, dtype=int)
        return [frontier[i] for i in indices]
    
    return frontier


def print_pick_tickets(frontier, bet_type=5, top_n=10):
    """Print top ticket options from the frontier."""
    sorted_f = sorted(frontier, key=lambda x: x[1], reverse=True)
    
    print(f"\n{'='*65}")
    print(f"TOP PICK {bet_type} TICKETS BY EXPECTED VALUE")
    print(f"{'='*65}")
    
    for i, (cost, ev, prob, selections) in enumerate(sorted_f[:top_n]):
        roi = ev / cost * 100 if cost > 0 else 0
        
        print(f"\nOption {i+1}: ${cost} cost | EV ${ev:+,.0f} | "
              f"Hit {prob:.2%} | ROI {roi:+.0f}%")
        print("-" * 50)
        
        for race_key in sorted(selections.keys()):
            race_num = int(race_key.split()[1])
            horses = selections[race_key]
            print(f"  R{race_num}: {', '.join(str(int(h)) for h in horses)}")


def print_race_probabilities(df, race_numbers, 
                              projected_prob_col='projected_win_probability',
                              ml_prob_col=None, blend_factor=0.5,
                              race_col='race', horse_col='horse_number'):
    """Print probability comparison table for pick ticket races."""
    
    has_ml = (ml_prob_col is not None and 
              ml_prob_col in df.columns and 
              df[ml_prob_col].notna().any())
    
    for race_num in race_numbers:
        race = df[df[race_col] == race_num].copy()
        
        if has_ml:
            race['blended'] = (blend_factor * race[projected_prob_col] + 
                               (1 - blend_factor) * race[ml_prob_col])
            total = race['blended'].sum()
            if total > 0:
                race['blended'] /= total
            race = race.sort_values('blended', ascending=False)
        else:
            race = race.sort_values(projected_prob_col, ascending=False)
        
        print(f"\n--- Race {race_num} ({len(race)} horses) ---")
        
        if has_ml:
            print(f"{'#':>3} {'Model':>7} {'ML':>7} {'Blend':>7}")
            for _, row in race.iterrows():
                print(f"{int(row[horse_col]):>3} "
                      f"{row[projected_prob_col]:>6.1%} "
                      f"{row[ml_prob_col]:>6.1%} "
                      f"{row['blended']:>6.1%}")
        else:
            print(f"{'#':>3} {'Model':>7}")
            for _, row in race.iterrows():
                print(f"{int(row[horse_col]):>3} {row[projected_prob_col]:>6.1%}")


# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    
    DATE = '4_4_26'
    TRACK = 'keeneland'
    STD_DEV = 5.8117
    DF = 1.1716
#     Optimal std_dev: 5.8117
# Optimal degrees_of_freedom: 1.1716
    # --- Build ensemble via pipeline (handles scratches + ML merge) ---
    # print("Building ensemble probabilities...")
    df = run_pipeline(
        DATE, track=TRACK,
        base_dir='../data/ai_picks',
        entries_dir='../data/entries',
        upc_dir='../data/upc_entries',
        std_dev=STD_DEV, degrees_of_freedom=DF,
        num_simulations=200000
    )
    
    # Determine ML column (if available)
    ml_col = 'ml_probability_normalized' if 'ml_probability_normalized' in df.columns else None
    blend = 0.55 if ml_col else 1.0
    
    # # --- Pick 5: Races 1-5 ---
    # print("\n" + "="*65)
    # print("EARLY PICK 5: RACES 1-5")
    # print("="*65)
    
    # early_races = [1, 2, 3, 4, 5]
    # print_race_probabilities(df, early_races, ml_prob_col=ml_col, blend_factor=blend)
    
    # frontier_early = optimize_pick_tickets(
    #     df, early_races,
    #     blend_factor=blend,
    #     ml_prob_col=ml_col,
    #     pool_size=80000,
    #     takeout_rate=0.22,
    #     max_cost=2500,
    #     min_total_prob=0.01,
    #     budget_steps=20
    # )
    # print_pick_tickets(frontier_early, bet_type=5, top_n=8)
    
    # --- Pick 5: Races 6-10 ---
    print("\n" + "="*65)
    print("LATE PICK 5: RACES 6-10")
    print("="*65)
    
    late_races = [7, 8, 9, 10, 11]
    print_race_probabilities(df, late_races, ml_prob_col=ml_col, blend_factor=blend)
    
    frontier_late = optimize_pick_tickets(
        df, late_races,
        blend_factor=blend,
        ml_prob_col=ml_col,
        pool_size=150000,  # Late pick 5 usually bigger pool
        takeout_rate=0.22,
        max_cost=2500,
        min_total_prob=0.005,
        budget_steps=20
    )
    print_pick_tickets(frontier_late, bet_type=5, top_n=8)
    
    # # --- Pick 3: Races 8-10 (Late Pick 3 at Keeneland) ---
    # print("\n" + "="*65)
    # print("LATE PICK 3: RACES 8-10")
    # print("="*65)
    
    # p3_races = [8, 9, 10]
    
    # frontier_p3 = optimize_pick_tickets(
    #     df, p3_races,
    #     blend_factor=blend,
    #     ml_prob_col=ml_col,
    #     pool_size=50000,
    #     takeout_rate=0.22,
    #     max_cost=500,
    #     min_total_prob=0.02,
    #     budget_steps=15
    # )
    # print_pick_tickets(frontier_p3, bet_type=3, top_n=8)

Loading AI picks for 4_4_26...
  112 entries across 11 races
Loading ML odds from upc_entries/keeneland.csv...
  Removed 23 scratches, 89 runners remain
Computing composite Beyer estimates...
Running MC simulation (n=200,000, std=5.8117, df=1.1716)...
Calculating edges...

Pipeline complete: 89 horses, 11 races
  Sources: ['claude_beyer', 'gemini_beyer']
  ML: True
  Results: False

LATE PICK 5: RACES 6-10

--- Race 7 (6 horses) ---
  #   Model      ML   Blend
  7  32.5%  36.6%  34.3%
  2  16.4%  23.8%  19.7%
  6  18.9%  10.6%  15.1%
  5  13.1%  17.3%  15.0%
  3  13.3%   7.3%  10.6%
  4   5.9%   4.5%   5.3%

--- Race 8 (10 horses) ---
  #   Model      ML   Blend
 10  30.1%  19.5%  25.3%
  9  16.3%  25.1%  20.3%
  8   9.0%  15.9%  12.1%
  7   9.1%   8.0%   8.6%
  6   7.8%   5.5%   6.7%
 11   7.4%   5.5%   6.6%
  4   6.2%   6.7%   6.4%
  1   4.9%   5.5%   5.2%
  3   4.8%   4.2%   4.5%
  2   4.3%   4.2%   4.2%

--- Race 9 (8 horses) ---
  #   Model      ML   Blend
  6  22.7%  19.5%  21.3%

In [86]:
df.loc[df['race']==6]

,race,horse_number,claude_beyer,gemini_beyer,date,track,ml_odds,horse_name,ml_probability_raw,ml_probability_normalized,beyer_disagreement,composite_beyer,projected_win_probability,edge,percentage_edge,kelly_stake,half_kelly,expected_roi
40,6,1,82.0,84.0,4_4_26,keeneland,20/1,Rosato,0.047619,0.050430,1.0,83.0,0.096085,0.045655,90.532411,0.048080,0.024040,0.905324
41,6,2,55.0,55.0,4_4_26,keeneland,30/1,Caberneigh,0.032258,0.034162,0.0,55.0,0.026945,-0.007217,-21.126002,0.000000,0.000000,-0.211260
42,6,3,96.0,105.0,4_4_26,keeneland,3/5,Shoot It True,0.625000,0.661890,4.5,100.5,0.623435,-0.038455,-5.809926,0.000000,0.000000,-0.058099
43,6,5,86.0,92.0,4_4_26,keeneland,6/1,Buttercream Babe,0.142857,0.151289,3.0,89.0,0.173670,0.022381,14.793373,0.026370,0.013185,0.147934
44,6,6,72.0,80.0,4_4_26,keeneland,12/1,Amalfi Drive,0.076923,0.081463,4.0,76.0,0.058055,-0.023408,-28.734891,0.000000,0.000000,-0.287349
45,6,9,45.0,60.0,4_4_26,keeneland,50/1,Letmenowhenurready,0.019608,0.020765,7.5,46.5,0.021810,0.001045,5.031560,0.001067,0.000533,0.050316
